In [ ]:
!pip install -q gradio google-genai gtts

In [ ]:
import time
import gradio as gr
from google import genai
from gtts import gTTS

API_KEY = "PASTE_YOUR_KEY_HERE"
MODELS = ["gemini-3.8-flash", "gemini-flash-latest", "gemini-flash-lite-latest"]


def build_prompt(age, characters, moral, language):
    age = int(age)
    if age <= 4:
        words, style = 100, "very simple words, short sentences, lots of repetition"
    elif age <= 7:
        words, style = 200, "simple words, short sentences, a little fun and surprise"
    else:
        words, style = 350, "slightly richer vocabulary, a small adventure with a clear plot"
    return (
        f"Write a bedtime story for a {age}-year-old child in {language}.\n"
        f"Characters: {characters}\n"
        f"Moral of the story: {moral}\n"
        f"Length: about {words} words. Use {style}.\n"
        "Give the story a short title, then the story, then one line starting with 'Moral:'.\n"
        "Keep it gentle, safe and positive. No violence or scary content."
    )


def ask_llm(prompt):
    client = genai.Client(api_key=API_KEY)
    last_error = None
    for model in MODELS:
        for attempt in range(3):
            try:
                resp = client.models.generate_content(model=model, contents=prompt)
                return resp.text
            except Exception as e:
                last_error = e
                msg = str(e)
                if "503" in msg or "429" in msg:
                    time.sleep(3 * (attempt + 1))  # wait, then retry
                    continue
                break  # other error (like 404): try next model
    raise last_error


def make_story(age, characters, moral, language):
    try:
        story = ask_llm(build_prompt(age, characters, moral, language))
    except Exception as e:
        return f"LLM ERROR: {e}", None

    try:
        lang = "hi" if language == "Hindi" else "en"
        gTTS(text=story, lang=lang).save("story.mp3")
        return story, "story.mp3"
    except Exception as e:
        return story + f"\n\n(Audio ERROR: {e})", None


demo = gr.Interface(
    fn=make_story,
    inputs=[
        gr.Slider(2, 10, value=5, step=1, label="Child's age"),
        gr.Textbox(value="a rabbit, a turtle", label="Characters"),
        gr.Textbox(value="Honesty is the best policy", label="Moral"),
        gr.Radio(["English", "Hindi"], value="English", label="Language"),
    ],
    outputs=[gr.Textbox(label="Story", lines=12), gr.Audio(label="Read aloud")],
    title="📖 Bedtime Story Generator for Kids",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://723fa6021e07ac29a4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
